# Démo du système complet : préparation des fichiers

**But** : préparer, dans ce dossier, tout ce qu'il faut pour faire **tourner en direct** le système complet (règle du contentieux puis modèle) sur des clients que le modèle n'a jamais vus, le jour de la présentation et sur Streamlit.

**Fichiers produits** (dans `lab_ML/demo_ML/`) :
- `clients_demo.csv` : les clients de la démo, avec les colonnes utiles à la règle, au modèle, et la cible `dpnm` (pour afficher le taux de défaut réel de chaque groupe) ;
- `model_ml_14.joblib` : copie du modèle final (CatBoost de `ml_14`, entraîné sur le train seul) ;
- `bornes_niveaux.json` : les seuils qui découpent les niveaux de risque (10 %, 30 % et 60 % des défauts détectés).

**Les clients de la démo** : exactement ceux de l'évaluation finale (`evaluation_finale_test.ipynb`, section 5) :
- les clients du **test** du ML (hors contentieux), jamais vus par le modèle ;
- un **échantillon de 20 % du contentieux à M** du même périmètre, tiré comme le test (stratifié sur la cible, `random_state=42`). La règle n'apprend rien : aucune fuite.

La part du contentieux est donc celle du périmètre.

**Les bornes** : calculées comme dans l'évaluation, sur les probabilités hors pli du **train** (`data/ML/probas_hors_pli/ml_14_v8.csv`), jamais sur le test. La borne de 60 % est le seuil de décision enregistré avec le modèle.

**Le test est relu ici sans rien décider** : ce notebook ne choisit ni modèle, ni seuil, ni borne. Il recopie et contrôle.

**Étapes** :
1. chargement ;
2. les clients de la démo ;
3. les bornes des niveaux de risque et la copie du modèle ;
4. contrôle : la copie du modèle, appliquée en direct, redonne le tableau de l'évaluation finale.

## 1. Chargement

In [1]:
import json
import shutil
import joblib
import numpy as np
import pandas as pd
from pathlib import Path
from IPython.display import Markdown, display
from sklearn.model_selection import train_test_split

current_dir = Path.cwd()
root_dir = next(p for p in [current_dir] + list(current_dir.parents) if (p / "data").exists())
dossier_ml = root_dir / "data" / "ML"
dossier_demo = root_dir / "lab_ML" / "demo_ML"
dossier_demo.mkdir(parents=True, exist_ok=True)

train = pd.read_csv(dossier_ml / "train_dataset.csv")
test = pd.read_csv(dossier_ml / "test_dataset.csv")
niveau_5 = pd.read_csv(root_dir / "data" / "cleaned5_creditcard.csv")
CHEMIN_MODELE = root_dir / "lab_ML" / "best_models" / "model_ml_14.joblib"
enregistre = joblib.load(CHEMIN_MODELE)
assert enregistre["scenario"] == "ml_14" and enregistre["nom"] == "CatBoost"
VARIABLES, SEUIL = enregistre["variables"], enregistre["seuil"]
oof = pd.read_csv(dossier_ml / "probas_hors_pli" / "ml_14_v8.csv").set_index("ID").loc[train["ID"]]
display(Markdown(f"**Modèle** : {enregistre['nom']} de `ml_14`, {len(VARIABLES)} variables, seuil de décision {SEUIL:.3f} ; "
                 f"**test** : {len(test)} clients ; **niveau 5** : {len(niveau_5)} clients"))

**Modèle** : CatBoost de `ml_14`, 21 variables, seuil de décision 0.492 ; **test** : 4840 clients ; **niveau 5** : 28851 clients

## 2. Les clients de la démo

Le contentieux à M (définition 2 : `FLAG_CTX` = 1 et `MOIS_SORTIE_CTX` = -1) est pris dans le périmètre (`BILL_AMT1 > 0`, `LIMIT_BAL <= 500 000`), puis tiré à 20 % comme dans l'évaluation finale. Les clients du contentieux n'ont pas `PAY_habituel` ni `CUMUL_INCIDENT` (colonnes calculées pour le ML, après le retrait du contentieux) : ces cases restent vides, car ces clients ne passent pas par le modèle.

In [2]:
perimetre = niveau_5[(niveau_5["BILL_AMT1"] > 0) & (niveau_5["LIMIT_BAL"] <= 500000)]
contentieux = perimetre[(perimetre["FLAG_CTX"] == 1) & (perimetre["MOIS_SORTIE_CTX"] == -1)]
_, ctx_demo = train_test_split(contentieux, test_size=0.2, stratify=contentieux["dpnm"], random_state=42)
assert not set(ctx_demo["ID"]) & set(test["ID"]), "Des clients du contentieux sont dans le test"
assert not ((test["FLAG_CTX"] == 1) & (test["MOIS_SORTIE_CTX"] == -1)).any(), "Le test contient du contentieux à M"

COLONNES = ["ID", "dpnm", "FLAG_CTX", "MOIS_SORTIE_CTX"] + VARIABLES
clients_demo = pd.concat([ctx_demo.reindex(columns=COLONNES), test[COLONNES]], ignore_index=True)
clients_demo.to_csv(dossier_demo / "clients_demo.csv", index=False)

au_ctx = (clients_demo["FLAG_CTX"] == 1) & (clients_demo["MOIS_SORTIE_CTX"] == -1)
nb_ml = len(train) + len(test)
display(Markdown(f"**Clients de la démo** : {len(clients_demo)}, dont {au_ctx.sum()} au contentieux à M "
                 f"({au_ctx.mean() * 100:.1f} %) et {(~au_ctx).sum()} hors contentieux. "
                 f"**Périmètre entier** : {len(contentieux)} au contentieux sur {len(contentieux) + nb_ml} clients "
                 f"({len(contentieux) / (len(contentieux) + nb_ml) * 100:.1f} %). Fichier : `clients_demo.csv`"))

**Clients de la démo** : 5441, dont 601 au contentieux à M (11.0 %) et 4840 hors contentieux. **Périmètre entier** : 3004 au contentieux sur 27202 clients (11.0 %). Fichier : `clients_demo.csv`

**Comment lire** : les deux parts du contentieux (démo et périmètre) doivent être presque égales : la démo garde la composition du portefeuille.

## 3. Les bornes des niveaux de risque et la copie du modèle

Pour chaque part des défauts visée, le seuil est la probabilité qui détecte cette part des défauts du train (probabilités hors pli de CatBoost), comme dans l'évaluation finale. Niveaux, du plus risqué au moins risqué : contentieux (règle), très haut risque (10 %), haut risque (30 %), risque modéré (60 %, seuil de décision), risque faible.

In [3]:
def seuil_pour_rappel(y, p, rappel):
    proba_defauts = np.sort(p[y == 1])[::-1]
    return proba_defauts[int(np.ceil(rappel * len(proba_defauts))) - 1]

y_train, p_train_oof = oof["dpnm"].to_numpy(), oof["CatBoost"].to_numpy()
bornes = {"tres_haut_risque": float(seuil_pour_rappel(y_train, p_train_oof, 0.10)),
          "haut_risque": float(seuil_pour_rappel(y_train, p_train_oof, 0.30)),
          "risque_modere": float(SEUIL)}
assert bornes["tres_haut_risque"] > bornes["haut_risque"] > bornes["risque_modere"]
(dossier_demo / "bornes_niveaux.json").write_text(json.dumps(bornes, indent=2), encoding="utf-8")
shutil.copy2(CHEMIN_MODELE, dossier_demo / "model_ml_14.joblib")
display(pd.Series(bornes, name="Probabilité à partir de laquelle le niveau commence").round(3).to_frame())

,Probabilité à partir de laquelle le niveau commence
tres_haut_risque,0.773
haut_risque,0.653
risque_modere,0.492


## 4. Contrôle : la démo en direct redonne l'évaluation finale

On relit les fichiers produits (et seulement eux), comme le fera la démo : la règle du contentieux d'abord, puis la copie du modèle sur les autres clients, puis les bornes. Le tableau doit être celui de l'évaluation finale (section 5 de `evaluation_finale_test.ipynb`, avec le haut risque coupé en deux par la borne de 10 %).

In [4]:
demo = pd.read_csv(dossier_demo / "clients_demo.csv")
copie = joblib.load(dossier_demo / "model_ml_14.joblib")
b = json.loads((dossier_demo / "bornes_niveaux.json").read_text(encoding="utf-8"))

regle = (demo["FLAG_CTX"] == 1) & (demo["MOIS_SORTIE_CTX"] == -1)          # étape 1 : la règle du contentieux
proba = pd.Series(np.nan, index=demo.index)
proba[~regle] = copie["modele"].predict_proba(demo.loc[~regle, copie["variables"]])[:, 1]   # étape 2 : le modèle, hors contentieux
assert np.allclose(proba[~regle], enregistre["modele"].predict_proba(test[VARIABLES])[:, 1]), "La copie ne redonne pas le modèle"

demo["niveau"] = np.select([regle, proba >= b["tres_haut_risque"], proba >= b["haut_risque"], proba >= b["risque_modere"]],
                           ["1. Contentieux (règle)", "2. Très haut risque", "3. Haut risque", "4. Risque modéré"], "5. Risque faible")
tableau = demo.groupby("niveau")["dpnm"].agg(Clients="size", Défauts="sum")
tableau["Taux de défaut réel (%)"] = (tableau["Défauts"] / tableau["Clients"] * 100).round(1)
tableau["Part de tous les défauts (%)"] = (tableau["Défauts"] / tableau["Défauts"].sum() * 100).round(1)
tableau["Défauts cumulés (%)"] = (tableau["Défauts"].cumsum() / tableau["Défauts"].sum() * 100).round(1)
tableau["Précision cumulée (%)"] = (tableau["Défauts"].cumsum() / tableau["Clients"].cumsum() * 100).round(1)
display(Markdown(f"**Système complet, calculé en direct sur les {len(demo)} clients de la démo** ({demo['dpnm'].sum()} défauts)"))
display(tableau)

**Système complet, calculé en direct sur les 5441 clients de la démo** (1195 défauts)

,Clients,Défauts,Taux de défaut réel (%),Part de tous les défauts (%),Défauts cumulés (%),Précision cumulée (%)
niveau,,,,,,
1. Contentieux (règle),601,424,70.5,35.5,35.5,70.5
2. Très haut risque,144,77,53.5,6.4,41.9,67.2
3. Haut risque,416,158,38.0,13.2,55.1,56.8
4. Risque modéré,1100,234,21.3,19.6,74.7,39.5
5. Risque faible,3180,302,9.5,25.3,100.0,22.0


**Comment lire** :
- le contentieux, le risque modéré et le risque faible doivent être identiques au tableau de l'évaluation finale ;
- le très haut risque et le haut risque, ensemble, doivent redonner son haut risque ;
- si c'est le cas, les fichiers de ce dossier suffisent à faire tourner la démo : la règle et le modèle calculent en direct, et retrouvent les chiffres du test, lu une seule fois.